# Can we predict malignant findings?
Target: `class` (1 = malignant). Six numeric features, no missing values.
The classes are very imbalanced (~2.3% malignant), so accuracy is meaningless (predicting "benign" always gives ~97.7%). We evaluate with **PR-AUC (average precision)**, ROC-AUC, and precision/recall at a chosen threshold, using stratified splits.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (average_precision_score, roc_auc_score, precision_recall_curve,
                             roc_curve, confusion_matrix, classification_report)
RS=0
df=pd.read_csv('data.csv')
print(df.shape, 'duplicates:', df.duplicated().sum())
df['class'].value_counts().to_frame('n').assign(share=lambda d:d.n/d.n.sum())

In [ ]:
df.describe().T

In [ ]:
X,y=df.drop(columns='class'),df['class']
fig,ax=plt.subplots(2,3,figsize=(13,6))
for a,c in zip(ax.ravel(),X.columns):
    lo,hi=X[c].quantile([.005,.995])
    for k,col in [(0,'tab:blue'),(1,'tab:red')]:
        a.hist(X.loc[y==k,c].clip(lo,hi),bins=40,alpha=.55,density=True,color=col,label=f'class {k}')
    a.set_title(c)
ax[0,0].legend(); plt.tight_layout(); plt.show()
X.corrwith(y).rename('corr with class').to_frame()

## Split
Hold out a stratified 25% test set; all model selection uses 5-fold stratified CV on the training part only.

In [ ]:
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,stratify=y,random_state=RS)
print(ytr.sum(), yte.sum())
cv=StratifiedKFold(5,shuffle=True,random_state=RS)
models={
 'baseline (prior)':DummyClassifier(strategy='prior'),
 'logreg (balanced)':make_pipeline(StandardScaler(),LogisticRegression(class_weight='balanced',max_iter=2000)),
 'random forest (balanced)':RandomForestClassifier(300,min_samples_leaf=3,class_weight='balanced_subsample',n_jobs=-1,random_state=RS),
 'hist gradient boosting':HistGradientBoostingClassifier(max_depth=4,learning_rate=.05,max_iter=300,class_weight='balanced',random_state=RS),
}
oof={}; rows=[]
for n,m in models.items():
    p=cross_val_predict(m,Xtr,ytr,cv=cv,method='predict_proba')[:,1]; oof[n]=p
    rows.append((n,average_precision_score(ytr,p),roc_auc_score(ytr,p)))
pd.DataFrame(rows,columns=['model','CV PR-AUC','CV ROC-AUC']).set_index('model').round(3)

The no-skill PR-AUC equals the prevalence (~0.023); anything well above that is real signal.

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(12,4.5))
for n,p in oof.items():
    pr,rc,_=precision_recall_curve(ytr,p); ax[0].plot(rc,pr,label=n)
    f,t,_=roc_curve(ytr,p); ax[1].plot(f,t,label=n)
ax[0].set(xlabel='recall',ylabel='precision',title='PR (CV, train)'); ax[1].set(xlabel='FPR',ylabel='TPR',title='ROC (CV, train)')
ax[1].plot([0,1],[0,1],'k:'); ax[0].legend(); plt.tight_layout(); plt.show()

## Choosing an operating point
The threshold depends on the cost of a missed malignancy vs. a false alarm, which we don't know. As a reasonable default, pick the threshold on the **training CV predictions** that maximises F2 (recall weighted above precision), then report on the untouched test set.

In [ ]:
best=max(rows[1:],key=lambda r:r[1])[0]; print('best by CV PR-AUC:',best)
p=oof[best]; pr,rc,th=precision_recall_curve(ytr,p)
f2=5*pr*rc/np.maximum(4*pr+rc,1e-12); i=np.nanargmax(f2[:-1]); thr=th[i]
print(f'threshold={thr:.3f}  CV precision={pr[i]:.3f} recall={rc[i]:.3f}')
final=models[best].fit(Xtr,ytr); pt=final.predict_proba(Xte)[:,1]
print(f'TEST PR-AUC={average_precision_score(yte,pt):.3f}  ROC-AUC={roc_auc_score(yte,pt):.3f}  (prevalence {yte.mean():.3f})')
pred=(pt>=thr).astype(int)
print(confusion_matrix(yte,pred)); print(classification_report(yte,pred,digits=3))

In [ ]:
# bootstrap uncertainty on the test PR-AUC (few positives in test)
rng=np.random.default_rng(RS); yt=yte.values; s=[]
for _ in range(500):
    j=rng.integers(0,len(yt),len(yt))
    if yt[j].sum(): s.append(average_precision_score(yt[j],pt[j]))
print('test PR-AUC 95%% CI: %.3f - %.3f'%tuple(np.percentile(s,[2.5,97.5])))

## Sensitivity: duplicate rows
~30% of rows are exact duplicates, almost all benign (only ~6 malignant rows are duplicates). Copies can fall on both sides of a split and flatter the score, so re-run the final model on de-duplicated data.

In [ ]:
d=df.drop_duplicates(); print(d['class'].value_counts().to_dict())
Xd,yd=d.drop(columns='class'),d['class']
Xa,Xb,ya,yb=train_test_split(Xd,yd,test_size=.25,stratify=yd,random_state=RS)
from sklearn.base import clone
pb=clone(models[best]).fit(Xa,ya).predict_proba(Xb)[:,1]
print(f'de-duplicated TEST PR-AUC={average_precision_score(yb,pb):.3f}  ROC-AUC={roc_auc_score(yb,pb):.3f}  (prevalence {yb.mean():.3f})')

In [ ]:
from sklearn.inspection import permutation_importance
r=permutation_importance(final,Xte,yte,scoring='average_precision',n_repeats=10,random_state=RS)
pd.Series(r.importances_mean,index=X.columns).sort_values().plot.barh(title='Permutation importance (test, PR-AUC drop)'); plt.show()